# Comparing LoRA initialization methods

Tachelhit → English translation on SmolLM2-1.7B
Comparing LoRA, PiSSA, and MiLoRA initialization strategies

## Setup

In [ ]:
# Clone the repos
!git clone --depth 1 https://github.com/fidel-mathenge/MiLoRA.git 2>/dev/null || echo 'MiLoRA already cloned'
!git clone --depth 1 https://github.com/GraphPKU/PiSSA.git 2>/dev/null || echo 'PiSSA already cloned'

import os
for repo in ['MiLoRA', 'PiSSA']:
    if os.path.isdir(repo):
        files = [f for f in os.listdir(repo) if not f.startswith('.')]
        print(f'\n{repo}/  →  {files[:10]}')
    else:
        print(f'{repo} not found — check URL')

In [ ]:
!pip install sacrebleu trl datasets transformers accelerate -q

# Remove peft if already installed (avoid cluster conflicts)
!pip uninstall -y peft

# Use the custom peft from MiLoRA
!pip install -e ./MiLoRA

## Imports

In [ ]:
import torch, torch.nn.functional as F, json, sacrebleu, contextlib, copy, gc
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from peft import LoraConfig, get_peft_model
import transformers
transformers.logging.set_verbosity_error()

MODEL_ID = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
RANK     = 16
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'gate_proj', 'up_proj', 'down_proj']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Load data

In [ ]:
# Load from file (upload to Colab if needed)
try:
    from google.colab import files
    uploaded = files.upload()   # pick en_tach_sentences.json
except ImportError:
    pass   # running locally — file must exist in CWD

with open('en_tach_sentences.json', encoding='utf-8') as f:
    data = json.load(f)

split      = int(len(data) * 0.8)
train_data = data[:split]
eval_data  = data[split:]
print(f'Total: {len(data)}  |  Train: {len(train_data)}  |  Eval: {len(eval_data)}')

sources = [d['input']  for d in eval_data]
targets = [d['output'] for d in eval_data]

## Load model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
print('Base model loaded.')

def translate(model, src):
    messages = [
        {'role': 'system',  'content': 'Translate the Tachelhiyt sentence to English. Output only the translation.'},
        {'role': 'user',    'content': src},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids  = tokenizer(text, return_tensors='pt').input_ids.to(DEVICE)
    plen = ids.shape[1]
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=64, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][plen:], skip_special_tokens=True).strip()

def evaluate(model, label=''):
    hyps = [translate(model, s) for s in sources]
    bleu = sacrebleu.corpus_bleu(hyps, [targets]).score
    chrf = sacrebleu.corpus_chrf(hyps, [targets]).score
    print(f'  [{label:10s}]  BLEU={bleu:.2f}  chrF={chrf:.2f}')
    return hyps, bleu, chrf

## 4. SVD Diagnostic on One Layer
Compare **which singular values** each method touches.

In [ ]:

layer_name = 'model.layers.0.self_attn.q_proj'
W = dict(base_model.named_modules())[layer_name].weight.float().detach().cpu()
print(f'Weight shape: {W.shape}')

U, S, Vh = torch.linalg.svd(W, full_matrices=False)
print(f'Number of singular values: {len(S)}')

# ── Explained variance curve ────────────────────────────────────────────────
# Plot the variance

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(S.numpy(), color='steelblue')
axes[0].axvspan(0,    RANK,   alpha=0.15, color='green',  label=f'PiSSA: top-{RANK}')
axes[0].axvspan(len(S)-RANK, len(S), alpha=0.15, color='orange', label=f'MiLoRA: bottom-{RANK}')
axes[0].set_title('Singular values (q_proj)')
axes[0].set_xlabel('Index'); axes[0].set_ylabel('σ')
axes[0].legend()

axes[1].plot(cumvar.numpy(), color='steelblue')
axes[1].axvline(RANK,         color='green',  linestyle='--', label=f'PiSSA cutoff (r={RANK})')
axes[1].axvline(len(S)-RANK,  color='orange', linestyle='--', label=f'MiLoRA cutoff')
axes[1].set_title('Cumulative variance')
axes[1].set_xlabel('Index'); axes[1].set_ylabel('Cumulative var (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('svd_analysis.png', dpi=150)
plt.show()

pissa_var  = cumvar[RANK-1].item()
milora_var = (1 - cumvar[len(S)-RANK-1]).item()
print(f'\nPiSSA  top-{RANK} captures  {pissa_var*100:.1f}% of weight energy')
print(f'MiLoRA last-{RANK} captures {milora_var*100:.1f}% of weight energy')
print('Insight:')
print('  PiSSA targets high-energy singular vectors (fast learning)')
print('  MiLoRA targets low-energy singular vectors (preserves base model)')

## Baseline

In [ ]:
print('Baseline evaluation...')
hyps_base, bleu_base, chrf_base = evaluate(base_model, label='Baseline')

In [ ]:
class ForgettingSFTTrainer(SFTTrainer):
    """
    SFTTrainer augmented with a KL-divergence regularization term that
    penalises deviation from the frozen base model's distribution.
    This is the 'forgetting loss' — it keeps the adapter from catastrophically
    overwriting pretrained knowledge.
    """
    def __init__(self, *args, forgetting_weight=0.1, forgetting_temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.forgetting_weight      = forgetting_weight
        self.forgetting_temperature = forgetting_temperature
        self._forget_history        = []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs   = model(**inputs)
        task_loss = outputs.loss

        # Frozen reference pass (base model = adapter disabled)
        ctx = model.disable_adapter() if hasattr(model, 'disable_adapter') else contextlib.nullcontext()
        with torch.no_grad(), ctx:
            ref_out = model(**inputs)

        T  = self.forgetting_temperature
        kl = F.kl_div(
            F.log_softmax(outputs.logits / T, dim=-1),
            F.softmax(ref_out.logits   / T, dim=-1),
            reduction='none',
        ).sum(dim=-1)

        mask   = inputs.get('attention_mask')
        if mask is None:
            labels = inputs.get('labels')
            mask   = (labels != -100).float() if labels is not None else torch.ones_like(kl)
        else:
            mask = mask.float()

        forget_loss = (kl * mask).sum() / mask.sum().clamp_min(1.0)
        loss = task_loss + self.forgetting_weight * (T ** 2) * forget_loss

        # Log for later analysis
        self._forget_history.append({
            'task_loss':   task_loss.item(),
            'forget_loss': forget_loss.item(),
            'total_loss':  loss.item(),
        })
        return (loss, outputs) if return_outputs else loss


def format_sample(d):
    messages = [
        {'role': 'system',    'content': 'Translate the Tachelhiyt sentence to English. Output only the translation.'},
        {'role': 'user',      'content': d['input']},
        {'role': 'assistant', 'content': d['output']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list([format_sample(d) for d in train_data])


def make_training_args(output_dir):
    return SFTConfig(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy='no',
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        dataset_text_field='text',
        max_length=512,
    )

## Training setup

## Run experiments

In [ ]:
import gc  # 🚨 Import nécessaire pour gc.collect()

METHODS = [
    # (name,  init_lora_weights value,  description)
    ('LoRA',   'gaussian', 'Random Gaussian init — classic LoRA'),
    ('PiSSA',  'pissa',    'Principal SVD subspace — high-energy directions'),
    ('MiLoRA', 'milora',   'Minor SVD subspace    — residual directions'),
]

results = {}

for method_name, init_weights, desc in METHODS:
    print(f'\n{"="*60}')
    print(f'  Method : {method_name}  |  {desc}')
    print(f'{"="*60}')

    # Load fresh base model
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map='auto'
    )

    lora_cfg = LoraConfig(
        r=RANK,
        lora_alpha=RANK,
        init_lora_weights=init_weights,
        target_modules=TARGET_MODULES,
        lora_dropout=0,
        bias='none',
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    trainer = ForgettingSFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        forgetting_weight=0.1,
        forgetting_temperature=2.0,
        args=make_training_args(f'./output_{method_name}'),
    )

    trainer.train()

    hyps, bleu, chrf = evaluate(model, label=method_name)

    # Compute metrics
    forget_history = trainer._forget_history
    avg_forget = np.mean([h['forget_loss'] for h in forget_history])
    avg_task   = np.mean([h['task_loss']   for h in forget_history])
    print(f'  Avg task_loss={avg_task:.4f}  |  Avg forget_loss(KL)={avg_forget:.4f}')

    results[method_name] = {
        'bleu': bleu, 'chrf': chrf,
        'delta_bleu': bleu - bleu_base,
        'delta_chrf': chrf - chrf_base,
        'avg_task_loss':   avg_task,
        'avg_forget_loss': avg_forget,
        'loss_history': forget_history,
        'hyps': hyps,
    }

    # Clean up
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

## Results

In [ ]:
print('\nResults:')
print(f'{"Method":12s} {"BLEU":>7} {"chrF":>7} {"Δ BLEU":>8} {"Δ chrF":>8} {"Forget":>10}')
print(f'{"Baseline":12s} {bleu_base:>7.2f} {chrf_base:>7.2f}')
for name, r in results.items():
    print(f'{name:12s} {r["bleu"]:>7.2f} {r["chrf"]:>7.2f}'
          f' {r["delta_bleu"]:>+8.2f} {r["delta_chrf"]:>+8.2f}'
          f' {r["avg_forget_loss"]:>10.4f}')

## Plots

In [ ]:
# BLEU and chrF scores
names  = list(results.keys())
bleus  = [r['bleu'] for r in results.values()]
chrfs  = [r['chrf'] for r in results.values()]

x = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vals, title, base in [
    (axes[0], bleus, 'BLEU Score', bleu_base),
    (axes[1], chrfs, 'chrF Score', chrf_base),
]:
    bars = ax.bar(x, vals, color=['#4C72B0','#55A868','#C44E52'])
    ax.axhline(base, color='gray', linestyle='--', label='Baseline')
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_title(title); ax.legend()
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.2, f'{v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('bleu_chrf_comparison.png', dpi=150)
plt.show()

# Forgetting loss over training
fig, ax = plt.subplots(figsize=(10, 4))
colors  = {'LoRA': '#4C72B0', 'PiSSA': '#55A868', 'MiLoRA': '#C44E52'}
for name, r in results.items():
    kls = [h['forget_loss'] for h in r['loss_history']]
    ax.plot(kls, label=name, color=colors.get(name, 'gray'), alpha=0.8)
ax.set_xlabel('Training step'); ax.set_ylabel('KL Forgetting Loss')
ax.set_title('Forgetting Loss (KL divergence from base) per Step')
ax.legend(); plt.tight_layout()
plt.savefig('forgetting_loss_curves.png', dpi=150)
plt.show()

## Examples

In [ ]:
N_SAMPLES = 5
print(f'Qualitative comparison — first {N_SAMPLES} eval sentences\n')
for i in range(N_SAMPLES):
    print(f'[{i+1}] Source    : {sources[i]}')
    print(f'    Reference : {targets[i]}')
    print(f'    Baseline  : {hyps_base[i]}')
    for name, r in results.items():
        print(f'    {name:8s}  : {r["hyps"][i]}')
    print()

---

## 11. Is BLEU + chrF + Forgetting Loss Enough? — Critical Analysis

### What your current metrics cover

| Metric | What it measures | Strength | Blind spot |
|--------|-----------------|----------|------------|
| **BLEU** | n-gram overlap (1–4) | Industry standard, fast | Punishes valid paraphrases; biased toward short outputs; saturates on low-resource data |
| **chrF** | Character n-gram F-score | Better for morphologically rich / agglutinative languages like Tachelhit | Still surface-level; no semantic understanding |
| **KL Forgetting Loss** | How much the adapter diverges from the base distribution | Directly measures catastrophic forgetting | Only measures distributional shift, not *which* knowledge is forgotten |

### What is missing and why it matters

**1. comet / bleurt (neural MT metrics)**  
BLEU and chrF are surface metrics. For a low-resource language like Tachelhit, where references may be imperfect, COMET (trained on human DA scores) catches meaning-preserving translations that BLEU punishes.  
```python
# pip install unbabel-comet
from comet import download_model, load_from_checkpoint
model_path = download_model('Unbabel/wmt22-comet-da')
comet_model = load_from_checkpoint(model_path)
data = [{'src': s, 'mt': h, 'ref': t} for s,h,t in zip(sources, hyps, targets)]
score = comet_model.predict(data, batch_size=8).system_score
```

**2. Perplexity on a held-out general corpus (forgetting depth)**  
KL divergence at token level doesn't tell you if the model has forgotten English grammar or world knowledge. Measure perplexity on WikiText-2 or a general English set before/after fine-tuning.
```python
from datasets import load_dataset
wiki = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
# compute perplexity → if it spikes, the adapter destroyed general capability
```

**3. SVD rank utilisation / effective rank of ΔW**  
After training, compute the effective rank of `B @ A` (the learned ΔW) for each method. MiLoRA should show *lower* effective rank (it works in the residual subspace), while PiSSA should show *higher* utilisation of its allocated rank.
```python
# After training
for name, param in model.named_parameters():
    if 'lora_A' in name:
        # find paired lora_B
        B = ...  # lora_B weight
        delta_W = B @ param
        _, S, _ = torch.linalg.svd(delta_W)
        eff_rank = (S / S[0] > 0.01).sum()  # 1% threshold
        print(f'{name}: effective rank = {eff_rank}')
```

**4. FLORES-200 zero-shot probe (generalisation beyond your dataset)**  
Your eval set is from the same distribution as training. A few FLORES-200 Tachelhit sentences (if available) or a held-out domain would test whether the adapter generalises.

**5. Calibration / confidence**  
Catastrophic forgetting also shows up as miscalibration. Check ECE (Expected Calibration Error) before/after. A forgetting KL of 0.0 is suspicious — it may mean the adapter is ignoring the regularisation.

### Recommended minimal addition

For a low-resource MT paper, the minimum credible evaluation is:
- **BLEU + chrF** ✅ (you have this)  
- **COMET** → semantic quality  
- **Perplexity on general English** → quantify forgetting depth  
- **Effective rank of ΔW** → understand what each SVD-init method actually learns  

BLEU + chrF + forgetting KL is a solid **engineering baseline**, but for a rigorous comparison of MiLoRA vs PiSSA you need at least COMET + general perplexity to make credible claims about the quality/forgetting tradeoff.

## Effective rank

In [ ]:
# Run this cell after the training loop above (model variable = last trained model)
# Reload the last model if needed, then:

def effective_rank(B, A, threshold=0.01):
    """Stable rank: sum(σ²) / σ_max² — measures how spread the update is."""
    dW = (B.float() @ A.float()).detach().cpu()
    _, S, _ = torch.linalg.svd(dW, full_matrices=False)
    stable_rank = (S**2).sum() / (S[0]**2 + 1e-12)
    hard_rank   = (S / (S[0] + 1e-12) > threshold).sum().item()
    return stable_rank.item(), hard_rank

print('Effective rank of ΔW = lora_B @ lora_A per module:\n')
print(f'{"Layer":45s} {"Stable rank":>12} {"Hard rank (1%)"}' )
print('-'*72)

lora_A_dict = {}
lora_B_dict = {}

for name, param in model.named_parameters():
    if 'lora_A' in name:
        lora_A_dict[name.replace('lora_A', '')] = param
    if 'lora_B' in name:
        lora_B_dict[name.replace('lora_B', '')] = param

for key in sorted(lora_A_dict.keys()):
    if key in lora_B_dict:
        A = lora_A_dict[key]
        B = lora_B_dict[key]
        sr, hr = effective_rank(B.weight if hasattr(B,'weight') else B,
                                A.weight if hasattr(A,'weight') else A)
        print(f'{key:45s} {sr:>12.2f} {hr:>12}')